# Grid Search Hyperparameter Tuning

## Hyperparameters vs Parameters — A Critical Distinction

**Parameters** are what the model *learns* from data — the weights in a neural network, the split thresholds in a decision tree.

**Hyperparameters** are configuration choices you make *before* training — the learning rate, the number of trees, the kernel type in SVM. The model cannot learn these from data; you have to set them.

The choice of hyperparameters can change accuracy from 70% to 93%. Getting them wrong is one of the most common reasons a model underperforms.

---

## The Problem: Which Values to Use?

For Kernel SVM there are at least two critical hyperparameters:

| Hyperparameter | What it controls |
|----------------|------------------|
| `C` | Regularisation strength. High C = hard margin (fits training data closely, risk of overfit). Low C = soft margin (more tolerant of misclassifications, risk of underfit). |
| `gamma` | RBF kernel bandwidth. High gamma = narrow kernel (complex, wiggly boundary). Low gamma = wide kernel (smooth boundary). |

Trying all combinations manually is impractical. And evaluating each configuration on a single train/test split is unreliable (high variance).

---

## The Solution: Grid Search + Cross Validation

Grid Search exhaustively tries **every combination** of hyperparameter values you specify. For each combination, it runs **k-fold cross-validation** to get a reliable performance estimate.

```
Grid of C x gamma:

         gamma=0.1  gamma=0.2  gamma=0.3  ... gamma=0.9
C=0.25    CV(10)     CV(10)     CV(10)        CV(10)
C=0.5     CV(10)     CV(10)     CV(10)        CV(10)
C=0.75    CV(10)     CV(10)     CV(10)        CV(10)
C=1.0     CV(10)     CV(10)     CV(10)        CV(10)
```

Each cell runs 10 training cycles. With 4 C values x 9 gamma values = 36 combinations x 10 folds = **360 model trainings** for just the RBF grid. This is why `n_jobs=-1` (use all CPU cores) is important.

---

## Grid Search vs Random Search

| Method | How it works | When to use |
|--------|-------------|-------------|
| **Grid Search** | Exhaustive — tries every combination in the grid | Small grids (2-3 hyperparameters, few values) |
| **Random Search** | Samples random combinations from distributions | Large grids (4+ hyperparameters) — finds good regions 10x faster |
| **Bayesian Search** | Builds a model of the search space, targets promising areas | Production tuning — most efficient |

Grid search is the right starting point for learning because its results are fully interpretable.

---

## What We Will Build

1. Train a Kernel SVM with default hyperparameters
2. Evaluate it with cross-validation (baseline)
3. Apply Grid Search to find the best `C` and `gamma`
4. Compare cross-validation accuracy before and after tuning

## Step 1: Import Libraries

| Library | Why we need it |
|---------|---------------|
| `numpy` | Array operations |
| `matplotlib` | Decision boundary visualisation |
| `pandas` | Loading the dataset |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Step 2: Load the Dataset

Same Social Network Ads dataset used in the k-Fold notebook:

- **400 samples**, 2 features (Age, EstimatedSalary)
- Binary target: purchased the product (1) or not (0)

A small dataset like this is ideal for demonstrating grid search because the training cycles complete quickly. On larger datasets, the computational cost of exhaustive grid search becomes prohibitive.

In [ ]:
dataset = pd.read_csv('Social_Network_Ads.csv')
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

## Step 3: Train/Test Split

We hold out 25% as a final test set that will not be used during hyperparameter tuning.

**This is critical:** Grid search uses cross-validation on the training set to select hyperparameters. The test set is only used for the final evaluation. If you used the test set to choose hyperparameters, you would be effectively training on it — your final accuracy estimate would be optimistic.

The principle: **hyperparameter selection uses training data only.**

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 0)

## Step 4: Feature Scaling

Required for Kernel SVM — distance-based model sensitive to feature scales.

Fit on training data only, transform both sets with the same scaler.

In [ ]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

## Step 5: Train With Default Hyperparameters (Baseline)

We train first with sklearn defaults: `C=1.0`, `gamma='scale'`. This gives us a baseline to compare against after grid search.

Default parameters are rarely optimal. They are designed to work reasonably well across many datasets, not to maximise performance on any specific one.

In [ ]:
from sklearn.svm import SVC
classifier = SVC(kernel = 'rbf', random_state = 0)
classifier.fit(X_train, y_train)

## Step 6: Baseline Evaluation

Single-split accuracy with default hyperparameters. Note this number carefully — we will see whether grid search improves on it.

Also run 10-fold cross-validation in the next step to get a more reliable baseline before tuning.

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
y_pred = classifier.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

## Step 7: Cross-Validated Baseline

10-fold CV on the default model gives us a reliable accuracy baseline with a standard deviation.

This is the number to beat with grid search. If grid search does not improve on it, the default hyperparameters were already near-optimal for this dataset.

In [ ]:
from sklearn.model_selection import cross_val_score
accuracies = cross_val_score(estimator = classifier, X = X_train, y = y_train, cv = 10)
print("Accuracy: {:.2f} %".format(accuracies.mean()*100))
print("Standard Deviation: {:.2f} %".format(accuracies.std()*100))

## Step 8: Apply Grid Search

We define a parameter grid with two sub-grids:

1. **Linear kernel:** Only `C` matters (no `gamma` for linear)
2. **RBF kernel:** Both `C` and `gamma`

Total combinations:
- Linear: 4 values of C
- RBF: 4 values of C x 9 values of gamma = 36 combinations
- **Total: 40 combinations x 10 folds = 400 model trainings**

`GridSearchCV` handles this automatically and returns:
- `best_score_`: the best cross-validated accuracy found
- `best_params_`: the exact hyperparameter values that produced it

**Why `n_jobs=-1`?** This tells sklearn to use all available CPU cores in parallel. Grid search is embarrassingly parallel — each combination is independent of the others. Without this, the 400 trainings run sequentially on one core.

In [ ]:
from sklearn.model_selection import GridSearchCV
parameters = [{'C': [0.25, 0.5, 0.75, 1], 'kernel': ['linear']},
              {'C': [0.25, 0.5, 0.75, 1], 'kernel': ['rbf'], 'gamma': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]}]
grid_search = GridSearchCV(estimator = classifier,
                           param_grid = parameters,
                           scoring = 'accuracy',
                           cv = 10,
                           n_jobs = -1)
grid_search.fit(X_train, y_train)
best_accuracy = grid_search.best_score_
best_parameters = grid_search.best_params_
print("Best Accuracy: {:.2f} %".format(best_accuracy*100))
print("Best Parameters:", best_parameters)

## Step 9: Visualise the Training Set Decision Boundary

The boundary plotted here uses the **original default model** (before grid search). If you want to visualise the optimised model, you would retrain with the best parameters from `grid_search.best_params_`.

The decision boundary shape is determined by both `C` and `gamma`: higher `gamma` creates a more curved, locally-fitted boundary; lower `gamma` creates a smoother, wider-margin boundary.

In [ ]:
from matplotlib.colors import ListedColormap
X_set, y_set = X_train, y_train
X1, X2 = np.meshgrid(np.arange(start = X_set[:, 0].min() - 1, stop = X_set[:, 0].max() + 1, step = 0.01),
                     np.arange(start = X_set[:, 1].min() - 1, stop = X_set[:, 1].max() + 1, step = 0.01))
plt.contourf(X1, X2, classifier.predict(np.array([X1.ravel(), X2.ravel()]).T).reshape(X1.shape),
             alpha = 0.75, cmap = ListedColormap(('red', 'green')))
plt.xlim(X1.min(), X1.max())
plt.ylim(X2.min(), X2.max())
for i, j in enumerate(np.unique(y_set)):
    plt.scatter(X_set[y_set == j, 0], X_set[y_set == j, 1],
                c = ListedColormap(('red', 'green'))(i), label = j)
plt.title('Kernel SVM (Training set)')
plt.xlabel('Age')
plt.ylabel('Estimated Salary')
plt.legend()
plt.show()

## Step 10: Visualise the Test Set Decision Boundary

Compare the single-split test accuracy here against the grid search cross-validation score.

**The takeaway from this notebook:**

1. Default hyperparameters are a starting point, not an endpoint
2. Grid search + cross-validation is the principled way to tune — exhaustive, evaluated reliably, not biased by a single split
3. The improvement from tuning can be modest (1-2%) or large (10%+) depending on how sensitive the model is to its hyperparameters
4. Always report the cross-validated score from grid search as your model performance — not the single test split, which is just one data point

In [ ]:
from matplotlib.colors import ListedColormap
X_set, y_set = X_test, y_test
X1, X2 = np.meshgrid(np.arange(start = X_set[:, 0].min() - 1, stop = X_set[:, 0].max() + 1, step = 0.01),
                     np.arange(start = X_set[:, 1].min() - 1, stop = X_set[:, 1].max() + 1, step = 0.01))
plt.contourf(X1, X2, classifier.predict(np.array([X1.ravel(), X2.ravel()]).T).reshape(X1.shape),
             alpha = 0.75, cmap = ListedColormap(('red', 'green')))
plt.xlim(X1.min(), X1.max())
plt.ylim(X2.min(), X2.max())
for i, j in enumerate(np.unique(y_set)):
    plt.scatter(X_set[y_set == j, 0], X_set[y_set == j, 1],
                c = ListedColormap(('red', 'green'))(i), label = j)
plt.title('Kernel SVM (Test set)')
plt.xlabel('Age')
plt.ylabel('Estimated Salary')
plt.legend()
plt.show()